# OpenPlaque — fixed proximal RCA + global graph extension

Self-contained notebook; no `%run`. This experiment retires the greedy deterministic BACCE surrogate.

The notebook freezes the previously accepted proximal RCA route as a Drive artifact, then extends from its distal end with a global 3-D graph search that forbids aortic re-entry, penalizes large-lumen structures, rewards coronary-scale tubular support, and evaluates many distal endpoints before choosing a route.


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-global-graph-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas


In [ ]:
import sys, shutil, json, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, SimpleITK as sitk
from scipy import ndimage as ndi
from skimage.filters import frangi
from skimage.graph import route_through_array
sys.path.insert(0,'/content/OpenPlaque/src')
from openplaque.study import OpenPlaqueStudy
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
OUT=ROOT/'RCA_Global_Graph'; OUT.mkdir(parents=True,exist_ok=True)
REFDIR=ROOT/'Fixed_RCA_Reference'; REFDIR.mkdir(parents=True,exist_ok=True)
REFCSV=REFDIR/'proximal_rca_reference_zyx.csv'
REFMETA=REFDIR/'proximal_rca_reference_metadata.json'


## 1. Load series 7 and cached TotalSegmentator aorta

In [ ]:
dz=ROOT/'Full_DICOM.zip'; lz=Path('/content/Full_DICOM.zip')
if not dz.exists(): raise FileNotFoundError(dz)
if not lz.exists() or lz.stat().st_size!=dz.stat().st_size: shutil.copyfile(dz,lz)
shutil.rmtree('/content/full_dicom_global_graph',ignore_errors=True)
study=OpenPlaqueStudy(str(lz),extract_root='/content/full_dicom_global_graph')
img,ct,_=study.load_series(7); ct=np.asarray(ct)
sp_xyz=np.array(img.GetSpacing(),float); sp_zyx=sp_xyz[::-1]
ap=ROOT/'RCA_Ostium_TotalSegmentator'/'aorta_series7_totalseg.nii.gz'
if not ap.exists(): raise FileNotFoundError(ap)
ai=sitk.ReadImage(str(ap))
if ai.GetSize()!=img.GetSize() or not np.allclose(ai.GetSpacing(),img.GetSpacing()): ai=sitk.Resample(ai,img,sitk.Transform(),sitk.sitkNearestNeighbor,0,sitk.sitkUInt8)
aorta=sitk.GetArrayFromImage(ai)>0
dist_aorta=ndi.distance_transform_edt(~aorta,sampling=sp_zyx)
print('shape z,y,x:',ct.shape,' spacing xyz:',tuple(sp_xyz))


## 2. Freeze the accepted proximal RCA reference

If the fixed reference already exists on Drive, it is loaded unchanged. Otherwise the exact accepted downstream-seed route formulation is reconstructed once, checked against the expected ~26-mm length, and saved permanently. Future runs never silently replace it.


In [ ]:
def route_arc(p):
    p=np.asarray(p,float)
    if len(p)<2:return np.array([0.])
    seg=np.linalg.norm(np.diff(p,axis=0)*sp_zyx,axis=1)
    return np.r_[0,np.cumsum(seg)]
if REFCSV.exists():
    ref=pd.read_csv(REFCSV)[['z','y','x']].to_numpy(float)
    print('Loaded FIXED proximal reference:',REFCSV)
else:
    prev=ROOT/'BACCE_Compatibility'/'bacce_compatibility_summary.csv'
    if not prev.exists(): raise FileNotFoundError('Need prior BACCE compatibility summary: '+str(prev))
    pr=pd.read_csv(prev).iloc[0]; ostium=np.array([pr.seed_z,pr.seed_y,pr.seed_x],float)
    o=np.round(ostium).astype(int); half=np.ceil(np.array([16.,34.,34.])/sp_zyx).astype(int)
    lo=np.maximum(0,o-half); hi=np.minimum(np.array(ct.shape),o+half+1)
    roi=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].astype(np.float32); ar=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]
    sl=ostium-lo; da=ndi.distance_transform_edt(~ar,sampling=sp_zyx)
    sm=ndi.gaussian_filter(roi,sigma=np.maximum(.65/sp_zyx,.5))
    zz=np.indices(ct.shape)[0]; a_hu=ct[aorta & (zz>=225) & (zz<=305)]
    med=float(np.median(a_hu)); blood_thr=float(np.clip(.43*med,180,360)); lo_hu=max(140.,.55*blood_thr); hi_hu=max(lo_hu+100.,1.35*med)
    ints=np.clip((sm-lo_hu)/(hi_hu-lo_hu),0,1); v=np.nan_to_num(frangi(sm,sigmas=(1,2,3),black_ridges=False))
    v99=np.percentile(v[v>0],99) if np.any(v>0) else 1.; vn=np.clip(v/max(v99,1e-6),0,1); support=.58*ints+.42*vn
    cost=1/(.06+support); cost[ar]+=40; cost[da<.6]+=15; cost[sm<lo_hu]+=12; cost=cost.astype(np.float32)
    z=int(round(ostium[0])); yy,xx=np.where(aorta[z]); a_ctr=np.array([z,float(np.mean(yy)),float(np.mean(xx))])
    axis_mm=(ostium-a_ctr)*sp_zyx; axis_mm/=np.linalg.norm(axis_mm)
    g=np.argwhere(np.ones_like(roi,bool)); dmm=(g-sl)*sp_zyx; rad=np.linalg.norm(dmm,axis=1); dot=dmm@axis_mm
    su=support[tuple(g.T)]; dd=da[tuple(g.T)]; m=(rad>=12)&(rad<=28)&(dot>=7)&(dd>=4)&(su>=np.percentile(support,82)); eg=g[m]
    if len(eg)==0: raise RuntimeError('No distal endpoints for fixed reference')
    es=support[tuple(eg.T)]+.02*da[tuple(eg.T)]; eps=[]
    for j in np.argsort(es)[::-1]:
        p=eg[j]
        if all(np.linalg.norm((p-q)*sp_zyx)>=3 for q in eps): eps.append(p)
        if len(eps)>=24: break
    def met(path):
        p=np.asarray(path,int); hu=roi[tuple(p.T)]; su=support[tuple(p.T)]; dd=da[tuple(p.T)]
        st=np.diff(p.astype(float),axis=0)*sp_zyx; seg=np.linalg.norm(st,axis=1); L=float(seg.sum())
        if len(st)>=2:
            u=st/np.maximum(np.linalg.norm(st,axis=1,keepdims=True),1e-6); turn=float(np.mean(np.arccos(np.clip(np.sum(u[:-1]*u[1:],axis=1),-1,1))))
        else: turn=np.pi
        outward=float(np.mean(np.diff(dd)>=-.35)) if len(dd)>1 else 0
        score=1.8*np.mean(su)+.45*outward+.018*min(L,25)+.025*min(float(dd[-1]),14)-.25*turn
        return score,L
    routes=[]; start=tuple(np.round(sl).astype(int))
    for ep in eps:
        try:p,_=route_through_array(cost,start,tuple(ep),fully_connected=True,geometric=True)
        except Exception:continue
        score,L=met(p)
        if 8<=L<=35:routes.append((score,np.asarray(p,int)))
    if not routes: raise RuntimeError('No usable fixed reference route')
    routes.sort(key=lambda x:x[0],reverse=True); ref=routes[0][1]+lo
    L=route_arc(ref)[-1]
    if not (23.0<=L<=29.5): raise RuntimeError(f'Reconstructed proximal route is {L:.2f} mm, outside accepted ~26-mm range; refusing to freeze it.')
    pd.DataFrame(ref,columns=['z','y','x']).to_csv(REFCSV,index=False)
    REFMETA.write_text(json.dumps({'length_mm':float(L),'source':'accepted BACCE downstream-seed route','immutable_after_creation':True},indent=2))
    print('Created FIXED proximal reference:',REFCSV)
ref=np.asarray(ref,float); ref_arc=route_arc(ref); ref_len=float(ref_arc[-1])
print('fixed reference length mm:',round(ref_len,2),' points:',len(ref))


## 3. Build global coronary-scale cost volume at the distal reference

In [ ]:
join_mm=max(12.0,ref_len-5.0); ji=int(np.argmin(np.abs(ref_arc-join_mm))); anchor=ref[ji]
w=np.where((ref_arc>=max(0,join_mm-6))&(ref_arc<=join_mm))[0]; P=ref[w]*sp_zyx; _,V=np.linalg.eigh(np.cov(P,rowvar=False)); tangent=V[:,-1]
if np.dot(tangent,(ref[ji]-ref[max(0,ji-4)])*sp_zyx)<0:tangent=-tangent
tangent/=np.linalg.norm(tangent)
half_mm=np.array([28.,55.,55.]); half=np.ceil(half_mm/sp_zyx).astype(int); o=np.round(anchor).astype(int)
lo=np.maximum(0,o-half); hi=np.minimum(np.array(ct.shape),o+half+1)
roi=ct[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]].astype(np.float32); ar=aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]; da=dist_aorta[lo[0]:hi[0],lo[1]:hi[1],lo[2]:hi[2]]; a=anchor-lo
sm=ndi.gaussian_filter(roi,sigma=np.maximum(.55/sp_zyx,.45))
ref_hu=ct[tuple(np.round(ref).astype(int).T)]; p10_ref=float(np.percentile(ref_hu,10)); med_ref=float(np.median(ref_hu))
lohu=max(120.,min(260.,.42*p10_ref)); hihu=max(lohu+120.,1.15*med_ref)
ints=np.clip((sm-lohu)/(hihu-lohu),0,1); v=np.nan_to_num(frangi(sm,sigmas=(0.7,1.0,1.4,1.8,2.2),black_ridges=False))
v99=np.percentile(v[v>0],99) if np.any(v>0) else 1.; vn=np.clip(v/max(v99,1e-6),0,1); support=.55*ints+.45*vn
bright=sm>=lohu; lumen_r=ndi.distance_transform_edt(bright,sampling=sp_zyx); large_pen=np.clip((lumen_r-3.8)/3.0,0,1)
cost=1/(.035+support)+8.0*large_pen; cost[sm<lohu]+=18; cost[ar | (da<1.5)]=1e5
G=np.indices(roi.shape).reshape(3,-1).T.astype(float); D=(G-a)*sp_zyx; forward=D@tangent; cost[(forward<-4).reshape(roi.shape)]+=25; cost=cost.astype(np.float32)
print('join ref arc',round(float(ref_arc[ji]),2),' ROI',roi.shape,' HU floor',round(lohu,1))


## 4. Search many distal endpoints globally and rank whole routes

In [ ]:
rad=np.linalg.norm(D,axis=1); su=support.ravel(); rr=lumen_r.ravel(); dd=da.ravel()
mask=(rad>=18)&(rad<=45)&(forward>=5)&(su>=np.percentile(support,88))&(rr<=4.8)&(dd>=2.0)
cand=G[mask].astype(int); vals=su[mask]+.02*np.minimum(dd[mask],15)-.08*np.maximum(rr[mask]-3,0)
if len(cand)==0: raise RuntimeError('No global graph endpoints')
order=np.argsort(vals)[::-1]; endpoints=[]
for j in order:
    p=cand[j]
    if all(np.linalg.norm((p-q)*sp_zyx)>=4 for q in endpoints): endpoints.append(p)
    if len(endpoints)>=80: break
def path_metrics(p):
    p=np.asarray(p,int); st=np.diff(p.astype(float),axis=0)*sp_zyx; seg=np.linalg.norm(st,axis=1); L=float(seg.sum())
    hu=roi[tuple(p.T)]; su=support[tuple(p.T)]; dr=lumen_r[tuple(p.T)]; dta=da[tuple(p.T)]
    if len(st)>=2:
        u=st/np.maximum(np.linalg.norm(st,axis=1,keepdims=True),1e-6); ang=np.arccos(np.clip(np.sum(u[:-1]*u[1:],axis=1),-1,1)); turn=float(np.mean(ang)); p95turn=float(np.percentile(ang,95))
    else:turn=p95turn=np.pi
    first=(p[min(len(p)-1,5)]-p[0])*sp_zyx; join_angle=float(np.degrees(np.arccos(np.clip(np.dot(first,tangent)/(np.linalg.norm(first)+1e-9),-1,1))))
    reentry=float(np.mean(dta<1.5)); large=float(np.mean(dr>4.8))
    score=(2.2*np.mean(su)+.50*np.percentile(su,10)+.018*min(L,38)+.025*min(float(dta[-1]),16)-.50*turn-.15*p95turn-.007*join_angle-1.6*reentry-1.1*large)
    return dict(score=float(score),length_mm=L,mean_hu=float(np.mean(hu)),p10_hu=float(np.percentile(hu,10)),mean_support=float(np.mean(su)),p10_support=float(np.percentile(su,10)),mean_radius_mm=float(np.mean(dr)),max_radius_mm=float(np.max(dr)),end_dist_aorta_mm=float(dta[-1]),mean_turn_rad=turn,p95_turn_rad=p95turn,join_angle_deg=join_angle,aorta_reentry_fraction=reentry,large_lumen_fraction=large)
routes=[]; start=tuple(np.round(a).astype(int))
for ep in endpoints:
    try:p,_=route_through_array(cost,start,tuple(ep),fully_connected=True,geometric=True)
    except Exception:continue
    m=path_metrics(p)
    if 15<=m['length_mm']<=55 and m['aorta_reentry_fraction']==0: routes.append({'path':np.asarray(p,int),**m})
if not routes: raise RuntimeError('No valid global routes')
routes=sorted(routes,key=lambda r:r['score'],reverse=True); display(pd.DataFrame([{k:v for k,v in r.items() if k!='path'} for r in routes[:12]]))
best=routes[0]; ext=best['path']+lo; print('BEST:',{k:round(v,3) for k,v in best.items() if k!='path'})


## 5. Join fixed reference to extension and save outputs

In [ ]:
prox=ref[:ji+1]; full=np.vstack([prox,ext[1:]]); full_arc=route_arc(full); full_len=float(full_arc[-1])
full_i=np.round(full).astype(int); hu=ct[tuple(full_i.T)]; dta=dist_aorta[tuple(full_i.T)]
landmarks={mm:int(np.argmin(np.abs(full_arc-mm))) for mm in [0,10,20,30,40,50,60] if mm<=full_len}
summary={**{k:v for k,v in best.items() if k!='path'},'fixed_reference_length_mm':ref_len,'join_reference_arc_mm':float(ref_arc[ji]),'full_route_length_mm':full_len,'full_mean_hu':float(np.mean(hu)),'full_p10_hu':float(np.percentile(hu,10)),'full_end_dist_aorta_mm':float(dta[-1]),'reached_50mm':bool(full_len>=50)}
display(pd.DataFrame([summary])); pd.DataFrame([summary]).to_csv(OUT/'global_graph_summary.csv',index=False)
pd.DataFrame(full,columns=['z','y','x']).assign(arc_mm=full_arc).to_csv(OUT/'global_graph_centerline_zyx.csv',index=False)


In [ ]:
z=int(round(np.median(full[:,0]))); z0=max(0,z-6);z1=min(ct.shape[0],z+7); mip=np.max(ct[z0:z1],axis=0)
fig,axs=plt.subplots(1,3,figsize=(18,6))
axs[0].imshow(mip,cmap='gray',vmin=-200,vmax=900); axs[0].plot(full[:,2],full[:,1],'-',lw=2,label='full route'); axs[0].plot(ref[:,2],ref[:,1],'.',ms=2,label='fixed proximal')
for mm,i in landmarks.items(): axs[0].plot(full[i,2],full[i,1],'o',ms=6); axs[0].text(full[i,2]+2,full[i,1],f'{mm} mm')
axs[0].legend(); axs[0].set_title(f'Axial MIP — full {full_len:.1f} mm'); axs[0].axis('off')
q=np.round(full).astype(int); loq=np.maximum(q.min(0)-8,0); hiq=np.minimum(q.max(0)+9,np.array(ct.shape)); box=ct[loq[0]:hiq[0],loq[1]:hiq[1],loq[2]:hiq[2]]
cor=np.max(box,axis=1); sag=np.max(box,axis=2)
axs[1].imshow(cor,cmap='gray',vmin=-200,vmax=900,aspect='auto'); axs[1].plot(full[:,2]-loq[2],full[:,0]-loq[0],'-',lw=2); axs[1].set_title('Local coronal MIP'); axs[1].axis('off')
axs[2].imshow(sag,cmap='gray',vmin=-200,vmax=900,aspect='auto'); axs[2].plot(full[:,1]-loq[1],full[:,0]-loq[0],'-',lw=2); axs[2].set_title('Local sagittal MIP'); axs[2].axis('off')
plt.tight_layout(); p=OUT/'01_global_route_orthogonal.png'; fig.savefig(p,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
fig,ax=plt.subplots(figsize=(12,5)); ax.plot(full_arc,hu,label='HU'); ax.axhline(lohu,ls='--',label='search HU floor'); ax2=ax.twinx(); ax2.plot(full_arc,dta,label='distance to aorta'); ax.set_xlabel('arc length from ostium (mm)'); ax.set_ylabel('HU'); ax2.set_ylabel('distance to aorta (mm)'); ax.set_title('Full-route QC'); fig.tight_layout(); p2=OUT/'02_global_route_qc.png'; fig.savefig(p2,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
print('Saved:',p,p2)


In [ ]:
marks=np.arange(0,min(full_len,65)+.01,5.0); inds=[int(np.argmin(np.abs(full_arc-m))) for m in marks]
n=len(inds); cols=4; rows=int(np.ceil(n/cols)); fig,axs=plt.subplots(rows,cols,figsize=(16,4*rows)); axs=np.atleast_1d(axs).ravel()
for ax in axs: ax.axis('off')
for ax,mm,i in zip(axs,marks,inds):
    z=int(round(full[i,0])); ax.imshow(ct[z],cmap='gray',vmin=-200,vmax=900); near=np.abs(full[:,0]-z)<=1.5; ax.plot(full[near,2],full[near,1],'.-',ms=3,lw=1); ax.plot(full[i,2],full[i,1],'o',ms=6); ax.set_title(f'{mm:.0f} mm, z={z}'); ax.axis('off')
plt.tight_layout(); p3=OUT/'03_dense_axial_sequence.png'; fig.savefig(p3,dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig); print('Saved:',p3)
